<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 실습 1. Zero-shot / One-shot / Few-shot 비교

### ① 라이브러리 설치

In [ ]:
# Hugging Face의 transformers 라이브러리를 설치합니다.
# -q 옵션은 설치 메시지를 간단하게 표시합니다.
!pip -q install transformers accelerate sentencepiece

### ② 모델 불러오기
한국어 instruction-following이 가능한 비교적 작은 모델을 사용합니다. Colab GPU 사용을 권장합니다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 사용할 모델 이름
# Colab에서 실행하기 쉽도록 비교적 작은 모델을 사용합니다.
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 문장을 토큰으로 변환하는 tokenizer를 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 사전학습된 언어모델을 불러옵니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("모델 로딩 완료!")

### ③ LLM에게 질문하는 함수

In [ ]:
def ask_llm(prompt):
    """
    prompt를 모델에 전달하고
    생성된 답변만 반환하는 함수
    """

    # 채팅 모델이 이해할 수 있는 message 형식으로 만듭니다.
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 모델의 chat template에 맞는 문자열로 변환합니다.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 문자열을 모델이 처리할 수 있는 token ID로 변환합니다.
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 모델이 답변을 생성합니다.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,

            # 최대 30개의 새 토큰을 생성합니다.
            max_new_tokens=30,

            # 실험 결과를 쉽게 비교하기 위해
            # 무작위 sampling을 사용하지 않습니다.
            do_sample=False
        )

    # 입력 부분을 제외하고 새로 생성된 답변만 가져옵니다.
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    # token ID를 다시 사람이 읽을 수 있는 문자열로 변환합니다.
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer.strip()

### 실험 A. Zero-shot
예제를 하나도 보여주지 않고 바로 문제를 해결하게 합니다.

In [ ]:
review = "배송은 조금 늦었지만 제품의 품질은 정말 만족스럽습니다."

zero_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

리뷰: {review}

감정:
"""

result = ask_llm(zero_shot_prompt)

print("=== Zero-shot ===")
print(result)

### 실험 B. One-shot
이번에는 모델에게 정답 예제 1개를 먼저 보여줍니다.

In [ ]:
one_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시]

리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(one_shot_prompt)

print("=== One-shot ===")
print(result)

### 실험 C. Few-shot
이번에는 서로 다른 유형의 예제 3개를 제공합니다.

In [ ]:
few_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시 1]
리뷰: 가격도 저렴하고 성능도 매우 좋습니다.
감정: 긍정

[예시 2]
리뷰: 특별히 좋지도 나쁘지도 않습니다.
감정: 중립

[예시 3]
리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(few_shot_prompt)

print("=== Few-shot ===")
print(result)